# Vector stores and semantic search



In [27]:
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [28]:
import torch

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata

    def __repr__(self):
        return f"Document(text={self.text}, metadata={self.metadata})"
        

class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.documents = []
        self.embeddings = None
        self.model = embedding_model

    def add_documents(self, documents: list[Document]):
        new_embeddings = self.model.encode(
            [doc.text for doc in documents],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

        self.documents.extend(documents)

    def get_n_documents(self, n: int) -> list[Document]:
        return self.documents[:n]

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        embeded_query = self.model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
        scores = embeded_query @ self.embeddings.T
        print(scores)

### Load dataset

In [29]:
import csv

In [30]:
documents = []

with open('data/animal-fun-facts-dataset.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for i, row in enumerate(reader):
        text = str(row.pop('text')) 
        documents.append(Document(text=text, metadata=row))

print(f"Loaded {len(documents)} documents.\n")

for i, doc in enumerate(documents):
    if (i >= 5):
        break
    print(f"Document {i+1}:")
    print(doc.text)
    print(doc.metadata)
    print()


Loaded 7734 documents.

Document 1:
Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 2:
Aardvarks
have rather primitive brains that are very small for the size of the
animal. Some have suggested they are not particularly bright....
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 3:
Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel.
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 4:
The aardvarks Latin family name "Tubulidentata" means "tube toothed"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-f

### Initialize Vector Store

In [31]:
model = SentenceTransformer("all-MiniLM-L6-v2")
store = VectorStore(model)
store.add_documents(documents)
print(model.device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4454.39it/s]


cuda:0


In [32]:
print(store.get_n_documents(3))

[Document(text=Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters", metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}), Document(text=Aardvarks
have rather primitive brains that are very small for the size of the
animal. Some have suggested they are not particularly bright...., metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}), Document(text=Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel., metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'})]


### Making queries

In [33]:
store.search("What is the fastest animal?")

tensor([0.2337, 0.3189, 0.0765,  ..., 0.2961, 0.2470, 0.3298], device='cuda:0')


## Part II: Filtering by metadata

In [34]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass